# BAG OF WORDS

## Getting the libraries and the dataset

In [1]:
import pandas as pd
import numpy as np

In [41]:
data_imdb = pd.read_csv(r'sentiment labelled sentences\imdb_labelled.txt',sep='\t')
data_imdb.columns = ['review_text','review_class']


data_amazon = pd.read_csv(r'sentiment labelled sentences\amazon_cells_labelled.txt',sep='\t')
data_amazon.columns = ['review_text','review_class']

data_yelp = pd.read_csv(r'sentiment labelled sentences\yelp_labelled.txt',sep='\t')
data_yelp.columns = ['review_text','review_class']



df = pd.concat([data_imdb,data_amazon,data_yelp])
df


,review_text,review_class
0,Not sure who was more lost - the flat characte...,0
1,Attempting artiness with black & white and cle...,0
2,Very little music or anything to speak of.,0
3,The best scene in the movie was when Gerardo i...,1
4,"The rest of the movie lacks art, charm, meanin...",0
...,...,...
994,I think food should have flavor and texture an...,0
995,Appetite instantly gone.,0
996,Overall I was not impressed and would not go b...,0
997,"The whole experience was underwhelming, and I ...",0


In [17]:
# !pip install nltk
import re
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [35]:
all_reviews = list()
lines = df['review_text'].values.tolist()
for text in lines:
    text = text.lower()

    #REMOVING all HTTP and HTTPS urls from the document
    pattern = re.compile(r'https?://(?:[a-zA-Z]|[0-9]|[$\-_@.&+]|[!*(),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
    text = pattern.sub('',text)
    #--------------------------------------------------------

    #REMOVING PUNCTUATIONS, NON ALPHABETS AND TOKENIZING
    tokens = word_tokenize(text)
    #Creates a mapping table configured to delete all standard punctuation marks
    table = str.maketrans('','', string.punctuation)
    #loops through the list of tokens and uses table to strip punctuation from each word
    stripped = [w.translate(table) for w in tokens]
    words = [word for word in stripped if word.isalpha()] #Keeps only pure alphabetical words, strippiong out numbers
    #--------------------------------------------------------
    # GETTING THE STEM VERSION OF WORDS AND REMOVING STOP WORDS
    stop_words = set(stopwords.words('english'))
    stop_words.discard('not')
    PS = PorterStemmer()#This is a tool that cuts words down to their stems
    words = [PS.stem(w) for w in words if not w in stop_words] 
    # Loops through the tokens and stems all words, while throwing away stop words
    #------------------------------------------------------------------------------------
    words = ' '.join(words)

    all_reviews.append(words)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
CV = CountVectorizer(min_df=3)
#Sets up a tool to count word frequencies across all documents in our corpus(all_reviews)
#mid_df=3 tells it to ignore rare words that appear in fewer than 3 reviews, this reduces noise

X = CV.fit_transform(all_reviews).toarray()
#Go through all documents in our corpus, create the vocabulary list and then vectorize each document


y = df['review_class']

print(np.shape(X))
print(np.shape(y))

(2745, 1196)
(2745,)


In [39]:
from sklearn.model_selection import train_test_split

X_train,X_test, y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=0)



from sklearn.naive_bayes import GaussianNB
model = GaussianNB()

model.fit(X_train, y_train)
y_pred = model.predict(X_test)



from sklearn.metrics import classification_report

print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.68      0.76      0.72       256
           1       0.77      0.69      0.73       293

    accuracy                           0.72       549
   macro avg       0.72      0.73      0.72       549
weighted avg       0.73      0.72      0.72       549

